##Test Full-precision WCE-SqueezeNet model on Kvasir ETIS-Larib dataset

In [ ]:
#!pip install scikit-learn
from torchvision import transforms
from PIL import Image
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
import pickle

train_dir = 'train'
val_dir = 'val'
test_dir = 'test'

class_names = ['normal', 'ulcerative_colitis', 'polyps', 'esophagitis']
num_classes = len(class_names)


train_images_p = []
train_labels = []
val_images_p = []
val_labels = []
test_images_p = []
test_labels = []

for label, class_name in enumerate(class_names):
    class_dir = os.path.join(train_dir, f'{label}_{class_name}')
    for image_name in os.listdir(class_dir):
      filename = os.path.join(class_dir, image_name)
      input_image = Image.open(filename)
      preprocess = transforms.Compose([
      transforms.Resize(256),
      transforms.CenterCrop(224),
      transforms.Resize(128),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
      ])
      input_tensor = preprocess(input_image)
      input_batch = input_tensor.unsqueeze(0)
      train_images_p.append(input_batch)
      train_labels.append(label)

for label, class_name in enumerate(class_names):
    class_dir = os.path.join(val_dir, f'{label}_{class_name}')
    for image_name in os.listdir(class_dir):
      filename = os.path.join(class_dir, image_name)
      input_image = Image.open(filename)
      preprocess = transforms.Compose([
      transforms.Resize(256),
      transforms.CenterCrop(224),
      transforms.Resize(128),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
      ])
      input_tensor = preprocess(input_image)
      input_batch = input_tensor.unsqueeze(0)
      val_images_p.append(input_batch)
      val_labels.append(label)

for label, class_name in enumerate(class_names):
    class_dir = os.path.join(test_dir, f'{label}_{class_name}')
    for image_name in os.listdir(class_dir):
      filename = os.path.join(class_dir, image_name)
      input_image = Image.open(filename)
      preprocess = transforms.Compose([
      transforms.Resize(256),
      transforms.CenterCrop(224),
      transforms.Resize(128),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
      ])
      input_tensor = preprocess(input_image)
      input_batch = input_tensor.unsqueeze(0)
      test_images_p.append(input_batch)
      test_labels.append(label)


In [ ]:
shape1 = test_images_p[0].shape[1]
shape2 = test_images_p[0].shape[2]
shape3 = test_images_p[0].shape[3]

test_images = torch.empty((len(test_images_p),shape1,shape2,shape3))
test_labels = torch.tensor(np.array(test_labels))

for i in range(len(test_images_p)):
    test_images[i] = test_images_p[i]

In [ ]:
import torch
import torchvision
from torch import nn
#from torchsummary import summary

squeezenet = torch.hub.load('pytorch/vision:v0.10.0', 'squeezenet1_0', pretrained=True)

squeezenet.classifier =   new_head = nn.Sequential( #replace imagenet classifier with custom classifier
    nn.Dropout(p=0.5, inplace=False),
    nn.AdaptiveAvgPool2d(output_size=1),
    nn.Flatten(1,-1),
    nn.Linear(in_features=512, out_features=4)
  )

squeezenet.to(torch.device('cpu'))
#dummy = torch.ones(3,128,128).to('cuda')
#summary(squeezenet,(3,128,128))
net = squeezenet

In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np

class Colon_Dataset(Dataset):
  def __init__(self, x, y):
    super(Colon_Dataset, self).__init__()
    self.input = x
    self.target = y

  def __getitem__(self,idx):
    return (
      self.input[idx].unsqueeze(0), #.astype(np.float32),
      self.target[idx]
    )

  def __len__(self):
    return len(self.target) # just one sample for this problem


# Create TensorDatasets
test_dataset = Colon_Dataset(test_images, test_labels)


# Create DataLoaders
batch_size = 1  # Batch size
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

In [ ]:
from sklearn.metrics import confusion_matrix, recall_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

device = torch.device('cpu')
loss = nn.CrossEntropyLoss()

def assess_model(net,dataloader):
  test_loss=0
  test_acc=0
  test_samples=0
  test_output = None
  test_labels = None
  raw_test_output = None

  with torch.no_grad():
        net.eval()
        for i,(test_data, test_targets) in  enumerate(dataloader):
            test_targets = test_targets.type(torch.LongTensor)
            test_data = test_data.squeeze(0).to(device)
            test_targets = test_targets.to(device)
            # Valid set forward pass
            test_out = net(test_data)
            # Test set loss
            batch_loss = loss(test_out, test_targets)
            test_loss += test_loss*test_targets.numel()
            test_samples += test_targets.numel()
            _, idx = test_out.max(1)
            if test_output is None:
              test_output = idx
              raw_test_output = test_out
              test_labels = test_targets
            else:
              test_output = np.append(test_output, idx, axis=0)
              raw_test_output = np.append(raw_test_output, test_out, axis=0)
              test_labels = np.append(test_labels, test_targets, axis=0)
            test_acc += np.sum((test_targets == idx).detach().cpu().numpy())
            #f.write(f'\n Test acc: {test_acc/test_samples}, Test loss: {test_loss/test_samples}')
            print(f'\r Assessment Acc: {test_acc/test_samples}, Assessment loss: {test_loss/test_samples}',end='')

        # Store loss history for future plotting
        test_loss = test_loss/test_samples
        test_acc = test_acc/test_samples

  test_labels = np.array(test_labels)
  test_output = np.array(test_output)
  # Calculate the confusion matrix
  conf_matrix = confusion_matrix(test_labels, test_output)

  print("Assessment Metrics")
  print("Confusion Matrix:")
  print(conf_matrix)

  from sklearn.metrics import recall_score

  # Compute recall (sensitivity) for each class
  sensitivity_per_class = recall_score(test_labels, test_output, average=None)

  # Print sensitivity for each class
  for i, sensitivity in enumerate(sensitivity_per_class):
    print(f"Sensitivity for class {i}: {sensitivity}")

  specificity_per_class = []

  for i in range(len(conf_matrix)):
    # True negatives: sum of all elements that are not in the i-th row or column
    TN = conf_matrix.sum() - (conf_matrix[i, :].sum() + conf_matrix[:, i].sum() - conf_matrix[i, i])
    # False positives: sum of the i-th column, excluding the diagonal element
    FP = conf_matrix[:, i].sum() - conf_matrix[i, i]
    # False negatives: sum of the i-th row, excluding the diagonal element
    FN = conf_matrix[i, :].sum() - conf_matrix[i, i]
    # True positives: diagonal element
    TP = conf_matrix[i, i]

    # Specificity
    specificity = TN / (TN + FP)
    specificity_per_class.append(specificity)

  # Print specificity for each class
  for i, specificity in enumerate(specificity_per_class):
    print(f"Specificity for class {i}: {specificity}")

  # Accuracy
  accuracy = accuracy_score(test_labels, test_output)

  # Classification error (loss)
  classification_error = 1 - accuracy
  print("Assessment Accuracy",accuracy)
  print(f"Classification Error (Loss): {classification_error}")
  return raw_test_output, test_output


In [ ]:
trained_folder = './'
net.load_state_dict(torch.load(trained_folder + f'/bestnetwork_fullprecision.pt', map_location=torch.device('cpu')))


print("Assess on test data")
raw_test_output, test_output = assess_model(net,test_dataloader)